# Demora entre inicio de síntomas y notificación de varicela en Sivigila, 2008–2025

Este cuaderno audita las cifras citadas en [la sección sobre el retraso de registro](02_rt_nacional.ipynb). Calcula, para cada registro incluido, `FEC_NOT − INI_SIN` en días. **La demora de notificación de un caso no es el intervalo serial entre dos casos vinculados por transmisión.**

La fuente son los [CSV individuales de Sivigila](https://github.com/thesis-varicella-colombia/data-processing/blob/main/data/raw/sivigila/README.md) preparados en el repositorio hermano `data-processing`. El [Parquet semanal de este repositorio](../data/sivigila_integrado_notification_semanal_municipal.parquet/) contiene categorías de demora, pero no las dos fechas individuales necesarias para calcular percentiles exactos. La [ficha de procesamiento](https://github.com/thesis-varicella-colombia/data-processing/blob/main/data/processed/sivigila/README.md) documenta los filtros y la corrección de fechas de 2013.

## 1. Fuente y criterios

El análisis selecciona los archivos cuyo **año de origen `ANO` es 2008–2025**. Este criterio sigue los CSV anuales; el año de `FEC_NOT` puede diferir cerca de los bordes del año. Se verifica el evento 831, se homologan tres códigos de residencia como en el [procesamiento original](https://github.com/thesis-varicella-colombia/data-processing/blob/main/scripts/processing/calculate_weekly_sivigila.py), se conservan entidades presentes en el DIVIPOLA local y se excluyen ajustes `6` y `D`. `FEC_NOT` se interpreta como día/mes/año; `INI_SIN` usa ese formato salvo en el CSV de 2013, donde usa mes/día/año. Las fechas no interpretables se excluyen del denominador de la demora. Se conservan demoras negativas para que el denominador incluya todos los pares de fechas interpretables; se informan por separado.

In [1]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'notebooks/02_rt_nacional.ipynb').exists():
    ROOT = ROOT.parent
assert (ROOT / 'notebooks/02_rt_nacional.ipynb').exists(), 'Ejecute desde la raíz o notebooks/'
SOURCE_ROOT = Path(os.environ.get('SIVIGILA_DATA_PROCESSING_ROOT', ROOT.parent / 'data-processing'))
CSV_DIR = SOURCE_ROOT / 'data/raw/sivigila/csv'
DIVIPOLA = SOURCE_ROOT / 'data/raw/municipios/DIVIPOLA-_Códigos_municipios_20261007.csv'
assert CSV_DIR.is_dir() and DIVIPOLA.is_file(), (CSV_DIR, DIVIPOLA)
print('Microdatos:', CSV_DIR)
print('DIVIPOLA:', DIVIPOLA)

Microdatos: /home/diego/repos/data-processing/data/raw/sivigila/csv
DIVIPOLA: /home/diego/repos/data-processing/data/raw/municipios/DIVIPOLA-_Códigos_municipios_20261007.csv


In [2]:
validos = set(pd.read_csv(DIVIPOLA, dtype=str, encoding='utf-8-sig', keep_default_na=False)['Código Municipio'].str.strip().str.zfill(5))
homologaciones = {'27086': '27493', '94663': '94343', '88000': '88001'}
columnas = ['ANO', 'COD_EVE', 'COD_DPTO_R', 'COD_MUN_R', 'AJUSTE', 'FEC_NOT', 'INI_SIN']

def interpretar_fecha(valores, formato):
    fecha = valores.str.extract(r'^(\d{1,2}/\d{1,2}/\d{4})', expand=False)
    return pd.to_datetime(fecha, format=formato, errors='coerce')

bloques = []
control = []
for anio in range(2008, 2026):
    archivo = CSV_DIR / f'Datos_{anio}_831.csv'
    if not archivo.is_file():
        raise FileNotFoundError(archivo)
    incluidos = pares_validos = 0
    for bloque in pd.read_csv(archivo, dtype=str, keep_default_na=False, encoding='utf-8-sig', usecols=columnas, chunksize=50_000):
        if not pd.to_numeric(bloque.ANO, errors='coerce').eq(anio).all():
            raise ValueError(f'ANO inesperado en {archivo.name}')
        if not pd.to_numeric(bloque.COD_EVE, errors='coerce').eq(831).all():
            raise ValueError(f'Evento distinto de 831 en {archivo.name}')
        codigo = (bloque.COD_DPTO_R.str.strip().str.zfill(2) + bloque.COD_MUN_R.str.strip().str.zfill(3)).replace(homologaciones)
        conservar = codigo.isin(validos) & ~bloque.AJUSTE.str.strip().str.upper().isin(['6', 'D'])
        bloque = bloque.loc[conservar]
        incluidos += len(bloque)
        notificacion = interpretar_fecha(bloque.FEC_NOT, '%d/%m/%Y')
        sintomas = interpretar_fecha(bloque.INI_SIN, '%m/%d/%Y' if anio == 2013 else '%d/%m/%Y')
        demora = (notificacion - sintomas).dt.days.dropna().astype('int32')
        pares_validos += len(demora)
        bloques.append(pd.DataFrame({'anio_origen': anio, 'demora_dias': demora.to_numpy()}))
    control.append({'anio_origen': anio, 'incluidos': incluidos, 'pares_con_fechas': pares_validos, 'sin_par_de_fechas': incluidos - pares_validos})

datos = pd.concat(bloques, ignore_index=True)
control = pd.DataFrame(control)
assert len(datos) == control.pares_con_fechas.sum()
display(control)

,anio_origen,incluidos,pares_con_fechas,sin_par_de_fechas
0,2008,63491,63491,0
1,2009,72950,72950,0
2,2010,73715,73715,0
3,2011,120772,120772,0
4,2012,100195,100180,15
5,2013,93606,37920,55686
6,2014,107924,107924,0
7,2015,110714,110714,0
8,2016,111984,111984,0
9,2017,86386,86386,0


## 2. Resumen nacional

Los percentiles se calculan sobre **todos los pares de fechas interpretables**, incluidas las demoras negativas. El porcentaje entre 0 y 3 días incluye ambos extremos; el de más de 7 días usa el mismo denominador. Estas proporciones describen la vigilancia, no la distribución de transmisión.

In [3]:
demora = datos.demora_dias
resumen = pd.DataFrame([{'pares_con_fechas': len(demora), 'p25_dias': demora.quantile(.25), 'mediana_dias': demora.median(), 'p75_dias': demora.quantile(.75), 'casos_0_a_3_dias': int(demora.between(0, 3).sum()), 'porcentaje_0_a_3': 100 * demora.between(0, 3).mean(), 'casos_mas_7_dias': int((demora > 7).sum()), 'porcentaje_mas_7': 100 * (demora > 7).mean(), 'demoras_negativas': int((demora < 0).sum())}])
display(resumen.round(2))

,pares_con_fechas,p25_dias,mediana_dias,p75_dias,casos_0_a_3_dias,porcentaje_0_a_3,casos_mas_7_dias,porcentaje_mas_7,demoras_negativas
0,1169382,1.0,2.0,4.0,788368,67.42,152368,13.03,3


## 3. Evolución de las demoras

La comparación 2009–2011 frente a 2019–2025 usa el año del CSV original (`ANO`) y el porcentaje de pares con `FEC_NOT − INI_SIN > 7`. La tabla anual permite comprobar qué años contribuyen a cada período.

In [4]:
anual = datos.groupby('anio_origen').demora_dias.agg(pares_con_fechas='size', mediana_dias='median', casos_mas_7_dias=lambda s: int((s > 7).sum()))
anual['porcentaje_mas_7'] = 100 * anual.casos_mas_7_dias / anual.pares_con_fechas
display(anual.round(2))

periodos = []
for nombre, inicio, fin in [('2009–2011', 2009, 2011), ('2019–2025', 2019, 2025)]:
    parte = datos.loc[datos.anio_origen.between(inicio, fin), 'demora_dias']
    periodos.append({'periodo': nombre, 'pares_con_fechas': len(parte), 'casos_mas_7_dias': int((parte > 7).sum()), 'porcentaje_mas_7': 100 * (parte > 7).mean()})
display(pd.DataFrame(periodos).round(2))

,pares_con_fechas,mediana_dias,casos_mas_7_dias,porcentaje_mas_7
anio_origen,,,,
2008,63491,3.0,12878,20.28
2009,72950,2.0,3279,4.49
2010,73715,2.0,3613,4.90
2011,120772,2.0,5283,4.37
2012,100180,2.0,13121,13.10
2013,37920,2.0,4870,12.84
2014,107924,2.0,14542,13.47
2015,110714,2.0,16085,14.53
2016,111984,2.0,16384,14.63


,periodo,pares_con_fechas,casos_mas_7_dias,porcentaje_mas_7
0,2009–2011,267437,12175,4.55
1,2019–2025,208364,36921,17.72


## Interpretación y límites

Una demora corta para muchos casos favorece la lectura de tendencias semanales de notificación, pero no demuestra que sus rezagos coincidan con el intervalo serial. El cambio en la fracción con demora superior a siete días indica que la distribución de retrasos no fue constante entre los períodos comparados. Las fechas pueden contener errores de registro; este análisis no identifica pares transmisor–caso secundario ni corrige esos errores. Tampoco cuantifica el sesgo que el retraso introduce en el estimador de `R_t`.